In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

#makes your graphs appear inside the notebook
%matplotlib inline
print("✅All libraries imported successfully!")



✅All libraries imported successfully!


In [3]:
df1 = pd.read_csv("../data/mexico-real-estate-1.csv")
df2 = pd.read_csv("../data/mexico-real-estate-2.csv")
df3 = pd.read_csv("../data/mexico-real-estate-3.csv")

print("df1 shape:", df1.shape)
print("df2 shape:", df2.shape)
print("df3 shape:", df3.shape)

df1 shape: (700, 6)
df2 shape: (700, 6)
df3 shape: (700, 5)


In [4]:
df3.head(10)



,property_type,place_with_parent_names,lat-lon,area_m2,price_usd
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71,48550.59
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233,168636.73
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300,86932.69
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,NaN,275,263432.41
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,"19.511938,-96.871956",84,68508.67
5,house,|México|Jalisco|Guadalajara|,"20.689157,-103.366728",175,102763.00
6,house,|México|Sinaloa|Mazatlán|,NaN,146,152421.99
7,apartment,|México|Tamaulipas|Tampico|,NaN,165,105397.95
8,apartment,|México|Distrito Federal|Miguel Hidalgo|,NaN,80,310923.97
9,house,|México|Yucatán|Mérida|,"21.0441317,-89.6216334",223,126477.54


In [5]:
print("=== FILE 1 ISSUES ===")
print("Columns:", df1.columns.tolist())
print("Data types:\n", df1.dtypes)
print("Missing values:\n", df1.isnull().sum())

print("\n=== FILE 2 ISSUES ===")
print("Columns:", df2.columns.tolist())
print("Data types:\n", df2.dtypes)
print("Missing values:\n", df2.isnull().sum())

print("\n=== FILE 3 ISSUES ===")
print("Columns:", df3.columns.tolist())
print("Data types:\n", df3.dtypes)
print("Missing values:\n", df3.isnull().sum())

=== FILE 1 ISSUES ===
Columns: ['property_type', 'state', 'lat', 'lon', 'area_m2', 'price_usd']
Data types:
 property_type        str
state                str
lat              float64
lon              float64
area_m2            int64
price_usd            str
dtype: object
Missing values:
 property_type      0
state              0
lat              117
lon              117
area_m2            0
price_usd          0
dtype: int64

=== FILE 2 ISSUES ===
Columns: ['property_type', 'state', 'lat', 'lon', 'area_m2', 'price_mxn']
Data types:
 property_type        str
state                str
lat              float64
lon              float64
area_m2            int64
price_mxn          int64
dtype: object
Missing values:
 property_type      0
state              0
lat              129
lon              129
area_m2            0
price_mxn          0
dtype: int64

=== FILE 3 ISSUES ===
Columns: ['property_type', 'place_with_parent_names', 'lat-lon', 'area_m2', 'price_usd']
Data types:
 property_type   

In [6]:
# Make a copy so we never destroy the original
df1_clean = df1.copy()

# --- FIX 1: price_usd column has "$" and "," making it a string ---
# Example: "$67,965.56" needs to become 67965.56 (a number)
df1_clean["price_usd"] = (
    df1_clean["price_usd"]
    .str.replace("$", "", regex=False)   # remove dollar sign
    .str.replace(",", "", regex=False)   # remove commas
    .astype(float)                        # convert to number
)

# --- FIX 2: Drop rows missing lat, lon, area, or price ---
df1_clean.dropna(subset=["lat", "lon", "area_m2", "price_usd"], inplace=True)

# --- FIX 3: Remove unrealistic outliers ---
df1_clean = df1_clean[df1_clean["area_m2"] > 10]
df1_clean = df1_clean[df1_clean["area_m2"] < 10000]
df1_clean = df1_clean[df1_clean["price_usd"] > 1000]
df1_clean = df1_clean[df1_clean["price_usd"] < 10_000_000]

# --- FIX 4: Reset index cleanly ---
df1_clean.reset_index(drop=True, inplace=True)

print("✅ df1 cleaned!")
print("Rows remaining:", len(df1_clean))
print(df1_clean.dtypes)
df1_clean.head()

✅ df1 cleaned!
Rows remaining: 583
property_type        str
state                str
lat              float64
lon              float64
area_m2            int64
price_usd        float64
dtype: object


,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.560181,-99.233528,150,67965.56
1,house,Nuevo León,25.688436,-100.198807,186,63223.78
2,apartment,Guerrero,16.767704,-99.764383,82,84298.37
3,apartment,Guerrero,16.829782,-99.911012,150,94308.80
4,house,Yucatán,21.052583,-89.538639,205,105191.37


In [8]:
df2_clean = df2.copy()

df2_clean["price_usd"] = (df2_clean["price_mxn"] / 19).round(2)

df2_clean.drop(columns=["price_mxn"], inplace=True)

df2_clean.dropna(subset=["area_m2", "price_usd"], inplace=True)

# --- FIX 3: Remove outliers ---
df2_clean = df2_clean[df2_clean["area_m2"] > 10]
df2_clean = df2_clean[df2_clean["area_m2"] < 10000]
df2_clean = df2_clean[df2_clean["price_usd"] > 1000]
df2_clean = df2_clean[df2_clean["price_usd"] < 10_000_000]

df2_clean.reset_index(drop=True, inplace=True)

print("✅ df2 cleaned!")
print("Rows remaining:", len(df2_clean))
df2_clean.head()    
    

✅ df2 cleaned!
Rows remaining: 700


,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.721081,-100.345581,72,68421.05
1,apartment,Puebla,NaN,NaN,190,131578.95
2,house,Morelos,23.634501,-102.552788,360,278947.37
3,house,Morelos,NaN,NaN,76,43157.89
4,house,Puebla,NaN,NaN,200,57894.74


In [9]:
df3_clean = df3.copy()

# --- FIX 1: Drop rows missing key columns ---
df3_clean.dropna(subset=["area_m2", "price_usd"], inplace=True)

# --- FIX 2: Make sure price_usd is a float (number) ---
df3_clean["price_usd"] = pd.to_numeric(df3_clean["price_usd"], errors="coerce")

# Drop any rows where conversion failed
df3_clean.dropna(subset=["price_usd"], inplace=True)

# --- FIX 3: Remove outliers ---
df3_clean = df3_clean[df3_clean["area_m2"] > 10]
df3_clean = df3_clean[df3_clean["area_m2"] < 10000]
df3_clean = df3_clean[df3_clean["price_usd"] > 1000]
df3_clean = df3_clean[df3_clean["price_usd"] < 10_000_000]

df3_clean.reset_index(drop=True, inplace=True)

print("✅ df3 cleaned!")
print("Rows remaining:", len(df3_clean))
df3_clean.head()

✅ df3 cleaned!
Rows remaining: 700


,property_type,place_with_parent_names,lat-lon,area_m2,price_usd
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71,48550.59
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233,168636.73
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300,86932.69
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,NaN,275,263432.41
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,"19.511938,-96.871956",84,68508.67


In [20]:
df3_clean.dropna(subset=["area_m2", "price_usd"], inplace=True)

df3_clean[["lat", "lon"]] = df3_clean["lat-lon"].str.split(",", expand=True)

df3_clean.drop(columns=["lat-lon"], inplace=True)

df3_clean.reset_index(drop=True, inplace=True)

print("✅df3 cleaned!")
print("Rows remaining:", len(df3_clean))
df3_clean.head()
      
      

✅df3 cleaned!
Rows remaining: 700


,property_type,place_with_parent_names,area_m2,price_usd,lat,lon
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,71,48550.59,19.52589,-99.151703
1,house,|México|Estado de México|Toluca|Metepec|,233,168636.73,19.2640539,-99.5727534
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,300,86932.69,19.268629,-99.671722
3,house,|México|Morelos|Temixco|Burgos Bugambilias|,275,263432.41,NaN,NaN
4,apartment,|México|Veracruz de Ignacio de la Llave|Veracruz|,84,68508.67,19.511938,-96.871956
